In [ ]:
library(here)
library(maplet)
library(dplyr)
library(purrr)

# set repo path
repo <- here()
renv::activate(project = repo)

In [ ]:
# load maplet object
D <- readRDS(here('data', 'preprocessed_venous_metabolon.RDS'))

In [ ]:
# mutate a new group that combines all non-group1 WSPH groups
D <- D %>% 
    mt_anno_mutate(anno_type="samples", col_name="who_group", term = case_when(
        f_wnowt %in% c('WSPH Group 2', 'WSPH Group 3', 'WSPH Group 4', 'WSPH Group 5') ~ 'WSPH Group 2-5',
        f_wnowt == 'WSPH Group 1' ~ 'WSPH Group 1',
        f_wnowt == 'No Group' ~ 'No Group',
        TRUE ~ 'AllElse'
    ))

# Moderation Effect Analysis

First, we'll define our confounders to correct our lms. 

In [ ]:
# Define confounders as a string
confounders <- "age + SEX + bmi"

# isolate metabolite names
met_names <- rownames(D)

First, we'll subset our maplet object to isolate RV-specific metabolite profiles that are non-missing. 

In [ ]:
D1_GLS <- D %>% 
    # exclude non-WSPH groups
    mt_modify_filter_samples(filter = who_group %in% c("WSPH Group 1", "WSPH Group 2-5")) %>% 
    # filter out samples with missing RVGLOB6n parameters
    mt_modify_filter_samples(filter = !is.na(RVGLOB6n))

In [ ]:
D1_FAC <- D %>% 
    # exclude non-WSPH groups
    mt_modify_filter_samples(filter = who_group %in% c("WSPH Group 1", "WSPH Group 2-5")) %>% 
    # filter out samples with missing RVFACn parameters
    mt_modify_filter_samples(filter = !is.na(RVFACn))

In [ ]:
D1_RVEF <- D %>% 
    # exclude non-WSPH groups
    mt_modify_filter_samples(filter = who_group %in% c("WSPH Group 1", "WSPH Group 2-5")) %>% 
    # filter out samples with missing mri_RVEF parameters
    mt_modify_filter_samples(filter = !is.na(mri_RVEF))

For our analysis, we'll need to prepare a df that contains the who-group assignment, metabolite profiles, RV parameter, and clinical patient confounder data (e.g. age, sex, bmi). 

In [ ]:
# function to prepare a df of who_groups, mets, and clin_var
interaction_df_prep <- function(D, clin_var) { 
    clin_df <- D %>% colData() %>% as.data.frame() %>% 
        # select the clinical variable
        select(!!sym(clin_var), who_group, age, SEX, bmi) %>% tibble::rownames_to_column('StudyID') %>% na.omit()

    assay_df <- D %>% assay() %>% t() %>% as.data.frame() %>% tibble::rownames_to_column('StudyID')

    # join these by StudyID
    joined_df <- clin_df %>% inner_join(assay_df, by = 'StudyID') %>% 
        tibble::column_to_rownames('StudyID')
}

In [ ]:
library(broom)
library(dplyr)
library(stringr)
library(purrr)

run_moderation_models <- function(data, clin_var, met_names, group_var = "who_group", covars = NULL) {
  
  # Ensure group_var only has the two groups of interest
  data <- data %>%
    filter(.data[[group_var]] %in% c("WSPH Group 1", "WSPH Group 2-5"))
  
  # Convert group_var to factor (if not already) and ensure two levels, Group 1 as reference
  data[[group_var]] <- factor(data[[group_var]], levels = c("WSPH Group 1", "WSPH Group 2-5"))

  results_list <- list()
  
  for (met in met_names) {
    
    # Construct the RHS of the formula
    interaction_term <- paste0(clin_var, "*", group_var)
    covar_str <- if (!is.null(covars) && length(covars) > 0) paste(covars, collapse = " + ") else NULL
    
    rhs <- paste(c(interaction_term, covar_str), collapse = " + ")
    formula_str <- paste0("`", met, "` ~ ", rhs)
    fmla <- as.formula(formula_str)
    
    # Fit the model
    model <- lm(fmla, data = data)
    
    # Tidy the results
    tidy_results <- broom::tidy(model)
    
    # Add metadata
    tidy_results <- tidy_results %>%
      mutate(
          met_name = met,
          
          # Coarse label (high-level term assignment)
          coefficient_type = case_when(
            term == "(Intercept)" ~ "Intercept",
            term == clin_var ~ "ClinicalVar",
            str_detect(term, group_var) & !str_detect(term, ":") ~ "Group",
            str_detect(term, ":") ~ "Interaction",
            TRUE ~ "Other"
          ),
          
          # Fine-grained label (low-level term assignment)
          coefficient_type_2 = case_when(
            term == "(Intercept)" ~ "Intercept",
            
            term == clin_var ~ "ClinicalVar",

            term == paste0(group_var, "WSPH Group 2-5") ~ "Group2345_Baseline",
            
            term == paste0(clin_var, ":", group_var, "WSPH Group 2-5") |
              term == paste0(group_var, "WSPH Group 2-5:", clin_var) ~ "Group2345_Interaction",
            
            term == "age" ~ "Other_age",
            term == "SEX" ~ "Other_SEX", 
            term == "bmi" ~ "Other_bmi",
            
            TRUE ~ "Other"
          )
        )
        
    # Store in list
    results_list[[met]] <- tidy_results
  }
  
  # Combine all results
  results_df <- bind_rows(results_list)
  
  return(results_df)
}

## Group 1 vs 2-5 Moderation Effect

### GLS 6`

In [ ]:
# define RV parameter
var_of_interest <- "RVGLOB6n"

# prepare df to interaction term anaylsis
joined_df <- interaction_df_prep(D1_GLS, clin_var = var_of_interest)

results_df <- run_moderation_models(
  data = joined_df, # df containing who_group, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names,  # metabolite names
  group_var = "who_group", # who_group variable
  covars = c("age", "SEX", "bmi") # confounders
)

# isolate the main association and the interaction term
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% 
    # correct for multiple comparisons
    mutate(FDR = p.adjust(p.value, method = 'fdr')) 

stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% 
    # correct for multiple comparisons
    mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, here('outputs', paste0(var_of_interest, '_moderation_group1_group2345_main.xlsx')))
write.csv(stats_interaction, here('outputs', paste0(var_of_interest, '_moderation_group1_group2345_interaction.xlsx')))

### FAC 

In [ ]:
# define RV parameter
var_of_interest <- "RVFACn"

# prepare df to interaction term anaylsis
joined_df <- interaction_df_prep(D1_FAC, clin_var = var_of_interest)

results_df <- run_moderation_models(
  data = joined_df, # df containing who_group, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names,  # metabolite names
  group_var = "who_group", # who_group variable
  covars = c("age", "SEX", "bmi") # confounders
)

# isolate the main association and the interaction term
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% 
    # correct for multiple comparisons
    mutate(FDR = p.adjust(p.value, method = 'fdr')) 

stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% 
    # correct for multiple comparisons
    mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, here('outputs', paste0(var_of_interest, '_moderation_group1_group2345_main.xlsx')))
write.csv(stats_interaction, here('outputs', paste0(var_of_interest, '_moderation_group1_group2345_interaction.xlsx')))

### RVEF

In [ ]:
# define RV parameter
var_of_interest <- "mri_RVEF"

# prepare df to interaction term anaylsis
joined_df <- interaction_df_prep(D1_RVEF, clin_var = var_of_interest)

results_df <- run_moderation_models(
  data = joined_df, # df containing who_group, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names,  # metabolite names
  group_var = "who_group", # who_group variable
  covars = c("age", "SEX", "bmi") # confounders
)

# isolate the main association and the interaction term
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% 
    # correct for multiple comparisons
    mutate(FDR = p.adjust(p.value, method = 'fdr')) 

stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% 
    # correct for multiple comparisons
    mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, here('outputs', paste0(var_of_interest, '_moderation_group1_group2345_main.xlsx')))
write.csv(stats_interaction, here('outputs', paste0(var_of_interest, '_moderation_group1_group2345_interaction.xlsx')))